# England Men's and Women's Test Matches

Download the available ball-by-ball Test match data from [Cricsheet](https://cricsheet.org/downloads/), keep matches involving England, and export separate men's and women's CSV files. This is Cricsheet's available coverage, not a complete historical record.

## 1. Imports and Configuration

Run the cells from top to bottom. Only Python's standard library is needed; no package installation is required. `OUTPUT_DIR` controls where the CSV files are saved.

The first code cell defines the conversion and runs small offline checks. Each output row is a delivery, including wides and no-balls. `over` uses Cricsheet's zero-based numbering; `delivery_in_over` counts recorded deliveries, not legal balls. `wickets_json` preserves all dismissal details, including multiple dismissals on one delivery.

In [ ]:
import csv
import io
import json
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

TEAM = 'England'
DATASETS = {
    'male': 'https://cricsheet.org/downloads/tests_male_json.zip',
    'female': 'https://cricsheet.org/downloads/tests_female_json.zip',
}
OUTPUT_DIR = Path.cwd() / 'cricsheet_exports'
FIELDS = [
    'match_id', 'gender', 'start_date', 'end_date', 'season', 'venue',
    'team_1', 'team_2', 'winner', 'result', 'innings', 'batting_team',
    'bowling_team', 'over', 'delivery_in_over', 'batter', 'non_striker',
    'bowler', 'runs_batter', 'runs_extras', 'runs_total', 'wides',
    'noballs', 'byes', 'legbyes', 'penalty', 'wicket_count', 'wickets_json',
]


def iter_deliveries(match, match_id):
    info = match['info']
    if TEAM not in info.get('teams', []) or info.get('match_type') != 'Test':
        return
    teams = info['teams']
    dates = info.get('dates', [])
    outcome = info.get('outcome', {})
    for innings_number, innings in enumerate(match.get('innings', []), start=1):
        for over in innings.get('overs', []):
            for delivery_number, delivery in enumerate(over.get('deliveries', []), start=1):
                extras = delivery.get('extras', {})
                wickets = delivery.get('wickets', [])
                yield {
                    'match_id': match_id,
                    'gender': info.get('gender', ''),
                    'start_date': dates[0] if dates else '',
                    'end_date': dates[-1] if dates else '',
                    'season': info.get('season', ''),
                    'venue': info.get('venue', ''),
                    'team_1': teams[0],
                    'team_2': teams[1],
                    'winner': outcome.get('winner', ''),
                    'result': outcome.get('result', ''),
                    'innings': innings_number,
                    'batting_team': innings['team'],
                    'bowling_team': next(team for team in teams if team != innings['team']),
                    'over': over['over'],
                    'delivery_in_over': delivery_number,
                    'batter': delivery['batter'],
                    'non_striker': delivery['non_striker'],
                    'bowler': delivery['bowler'],
                    'runs_batter': delivery['runs']['batter'],
                    'runs_extras': delivery['runs']['extras'],
                    'runs_total': delivery['runs']['total'],
                    **{name: extras.get(name, 0) for name in ['wides', 'noballs', 'byes', 'legbyes', 'penalty']},
                    'wicket_count': len(wickets),
                    'wickets_json': json.dumps(wickets, ensure_ascii=False),
                }

sample = {
    'info': {'teams': ['England', 'Australia'], 'match_type': 'Test', 'gender': 'female'},
    'innings': [{'team': 'England', 'overs': [{'over': 0, 'deliveries': [
        {'batter': 'Batter', 'non_striker': 'Partner', 'bowler': 'Bowler',
         'runs': {'batter': 0, 'extras': 1, 'total': 1}, 'extras': {'wides': 1}},
        {'batter': 'Batter', 'non_striker': 'Partner', 'bowler': 'Bowler',
         'runs': {'batter': 0, 'extras': 0, 'total': 0},
         'wickets': [{'player_out': 'Batter', 'kind': 'bowled'}]},
    ]}]}],
}
sample_rows = list(iter_deliveries(sample, 'sample'))
assert len(sample_rows) == 2
assert sample_rows[0]['wides'] == 1
assert sample_rows[1]['wicket_count'] == 1
assert sample_rows[1]['delivery_in_over'] == 2
assert not list(iter_deliveries({**sample, 'info': {**sample['info'], 'teams': ['India', 'Australia']}}, 'other'))
assert not list(iter_deliveries({**sample, 'info': {**sample['info'], 'match_type': 'ODI'}}, 'odi'))
print('Offline checks passed: England-only Tests, extras and wickets.')

## 2. Download, Filter and Export

The next cell downloads the men's and women's Test archives separately, reads JSON directly from each ZIP without extracting files, and filters for England Test matches. It writes one CSV per gender:

- `england_male_tests_deliveries.csv`
- `england_female_tests_deliveries.csv`

Files are saved inside `cricsheet_exports` under the kernel's working directory; the cell prints their absolute paths. Internet access is required, and downloads may take a little time. Only completed, nonempty exports replace existing CSVs.

## 3. Data Quality and Preview

The same cell checks run totals during conversion, then reopens both CSVs to validate gender, England participation and unique delivery identifiers, and prints three sample rows per file. Match IDs repeat intentionally because each match contains many deliveries. Missing outcome fields are left blank; dismissal records are retained as JSON text in `wickets_json`.

Data source and attribution: [Cricsheet](https://cricsheet.org/). Consult the source's coverage and usage information before redistribution. These files contain the matches currently available from Cricsheet, not every historical England Test.

In [ ]:
def export_dataset(gender, url):
    output_path = OUTPUT_DIR / f'england_{gender}_tests_deliveries.csv'
    temporary_path = output_path.with_suffix('.csv.part')
    request = Request(url, headers={'User-Agent': 'EnglandTestCSVNotebook/1.0'})
    print(f'Downloading {gender} Test matches from {url}', flush=True)
    with urlopen(request, timeout=120) as response:
        archive_bytes = response.read()
    match_ids = set()
    delivery_count = 0
    try:
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            with temporary_path.open('w', newline='', encoding='utf-8') as output:
                writer = csv.DictWriter(output, fieldnames=FIELDS)
                writer.writeheader()
                for name in sorted(archive.namelist()):
                    if not name.endswith('.json'):
                        continue
                    with archive.open(name) as source:
                        match = json.load(source)
                    version = str(match.get('meta', {}).get('data_version', ''))
                    if version.split('.')[0] != '1':
                        raise ValueError(f'Unsupported Cricsheet JSON version: {version}')
                    if match['info'].get('gender') != gender:
                        continue
                    match_id = Path(name).stem
                    for row in iter_deliveries(match, match_id):
                        assert row['runs_total'] == row['runs_batter'] + row['runs_extras']
                        writer.writerow(row)
                        delivery_count += 1
                        match_ids.add(match_id)
        if delivery_count == 0:
            raise ValueError(f'No England {gender} Test deliveries found; no CSV replaced.')
        temporary_path.replace(output_path)
    finally:
        temporary_path.unlink(missing_ok=True)
    print(f'{len(match_ids):,} matches; {delivery_count:,} deliveries -> {output_path.resolve()}')
    return output_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exported_files = {
    gender: export_dataset(gender, url)
    for gender, url in DATASETS.items()
}

for gender, path in exported_files.items():
    with path.open(newline='', encoding='utf-8') as source:
        reader = csv.DictReader(source)
        preview = []
        keys = set()
        for row in reader:
            assert TEAM in (row['team_1'], row['team_2'])
            assert row['gender'] == gender
            key = (row['match_id'], row['innings'], row['over'], row['delivery_in_over'])
            assert key not in keys, f'Duplicate delivery: {key}'
            keys.add(key)
            if len(preview) < 3:
                preview.append(row)
        assert preview, f'Empty CSV: {path}'
    print(f'Validated {len(keys):,} unique deliveries in {path.name}')
    for row in preview:
        print({name: row[name] for name in ['match_id', 'start_date', 'batting_team', 'batter', 'runs_total']})